In [ ]:
# Experiment No. 9
# Title: Triplet Loss using TensorFlow and Keras

# Aim:
# To implement Triplet Loss using TensorFlow and Keras
# for learning similarity between image samples.

# Import Required Libraries
import tensorflow as tf
import numpy as np

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Flatten
from tensorflow.keras.layers import Lambda

# Parameters
embedding_dim = 32
input_shape = (28, 28)

# Embedding Network
def create_embedding_network():

    inputs = Input(shape=input_shape)

    x = Flatten()(inputs)

    x = Dense(
        64,
        activation='relu'
    )(x)

    x = Dense(
        32,
        activation='relu'
    )(x)

    outputs = Dense(embedding_dim)(x)

    return Model(inputs, outputs)

# Create Embedding Model
embedding_network = create_embedding_network()

# Inputs
anchor_input = Input(shape=input_shape)
positive_input = Input(shape=input_shape)
negative_input = Input(shape=input_shape)

# Embeddings
anchor_embedding = embedding_network(anchor_input)
positive_embedding = embedding_network(positive_input)
negative_embedding = embedding_network(negative_input)

# Distance Function
def euclidean_distance(vectors):

    x, y = vectors

    return tf.reduce_sum(
        tf.square(x - y),
        axis=1,
        keepdims=True
    )

# Calculate Distances
positive_distance = Lambda(
    euclidean_distance
)([anchor_embedding, positive_embedding])

negative_distance = Lambda(
    euclidean_distance
)([anchor_embedding, negative_embedding])

# Triplet Difference
output = Lambda(
    lambda tensors: tensors[0] - tensors[1]
)([positive_distance, negative_distance])

# Build Model
triplet_model = Model(
    inputs=[
        anchor_input,
        positive_input,
        negative_input
    ],
    outputs=output
)

# Triplet Loss Function
def triplet_loss(y_true, y_pred):

    margin = 1.0

    return tf.reduce_mean(
        tf.maximum(y_pred + margin, 0.0)
    )

# Compile Model
triplet_model.compile(
    optimizer='adam',
    loss=triplet_loss
)

# Generate Small Dummy Dataset
num_samples = 40

anchor_data = np.random.random(
    (num_samples, 28, 28)
)

positive_data = np.random.random(
    (num_samples, 28, 28)
)

negative_data = np.random.random(
    (num_samples, 28, 28)
)

dummy_labels = np.zeros((num_samples, 1))

# Train Model
history = triplet_model.fit(
    [
        anchor_data,
        positive_data,
        negative_data
    ],
    dummy_labels,
    epochs=3,
    batch_size=8
)

print("\nTriplet Loss Model Trained Successfully")

# Conclusion
# Successfully implemented Triplet Loss using TensorFlow
# and Keras for similarity learning.

Epoch 1/3
5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 43ms/step - loss: 1.0784
Epoch 2/3
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - loss: 0.1505
Epoch 3/3
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - loss: 2.7397e-04

Triplet Loss Model Trained Successfully
